[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P10/P10-1_Calcul_Symbolique.ipynb)

# Calcul symbolique avec SymPy

Chapitre 10 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études écrit la flèche d'une poutre, la dérive pour savoir quelle dimension compte le plus, l'inverse pour trouver la longueur qui tient une flèche admissible, puis recopie le résultat dans un programme. À chaque recopie, une puissance, un signe ou une conversion peut se glisser dans la formule. Vous voudriez que l'ordinateur manipule la formule elle-même, en lettres, et ne la change en nombres qu'à la fin, avec une valeur de référence exacte pour contrôler vos calculs ?

**Objectif terminal.** Ce chapitre montre comment, avec le calcul symbolique de la bibliothèque SymPy.

**Objectif instrumental.**

- Écrire une formule avec des symboles, la simplifier, la dériver, l'intégrer.
- Résoudre en lettres une équation, un système et une équation différentielle.
- Écrire une équation caractéristique avec une matrice littérale.
- Transformer une expression en fonction NumPy et la confronter à une valeur exacte.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 6, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 10.1 | La flèche d'une poutre en lettres | 15 min |
| 10.2 | Arbre en torsion et deux masses | 20 min |
| 10.3 | Fréquences propres et treillis (défi) | 30 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("sympy", version("sympy"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [ ]:
import math
import numpy as np
import sympy as sp

## Exercice 10.1 : La flèche d'une poutre en lettres (15 min)

Écrivez en lettres la flèche d'une poutre sur deux appuis chargée en son milieu, $f = F L^3/(48\,E\,I)$, avec $I = b\,h^3/12$, et simplifiez-la. Remplacez les lettres par les valeurs de l'exercice du chapitre 3 ($F = 2\,000$ N, $L = 1\,000$ mm, $E = 210\,000$ MPa, $b = 20$ mm, $h = 40$ mm), gardez la valeur exacte, puis chiffrez-la. Dérivez enfin $f$ par rapport à $h$ et calculez la variation relative de la flèche pour une variation relative de $h$.

**Résultat attendu.** Flèche simplifiée $f = \frac{F L^{3}}{4 E b h^{3}}$ ; valeur exacte $\frac{625}{336}$ mm, soit 1,8601 mm ; dérivée $- \frac{3 F L^{3}}{4 E b h^{4}}$ ; la flèche varie comme $h^{-3}$.

**Seconde voie.** La fonction du chapitre 3 (section 3.2) redonne 1,86 mm, et le rapport des flèches de $30$ et de $60$ mm vaut huit, soit $2^3$.

**Indication.** Déclarez les lettres positives, posez le moment quadratique $b\,h^3/12$ et simplifiez la flèche. La méthode `subs`, avec un dictionnaire, donne la fraction exacte, `evalf` sa valeur décimale. La variation relative est $h\,f'/f$, que `simplify` réduit à $-3$.

In [ ]:
# Votre code :
assert abs(fleche_mm - 1.8601) < 5e-5, "flèche attendue : 1,8601 mm"
# Votre code :
assert abs(fleche_mm - fleche_flottants) < 1e-12, "formule du chapitre 3"
# Votre code :
assert sensibilite == -3, "la flèche varie comme h à la puissance moins trois"
assert abs(dfleche_dh_num - (-3 * fleche_mm / 40)) < 1e-12
# Votre code :
assert sp.simplify(-y_milieu - fleche_brute) == 0, "l'intégration doit redonner F L**3 / (48 E I)"
# Votre code :
assert sp.expand(kt_developpe - kt_heywood) == 0 and kt_developpe.subs(r, 0) == 3
assert sp.expand(I_tube_facteurs - I_tube) == 0
print("Exercice 10.1 : contrôles passés")

## Exercice 10.2 : Arbre en torsion et deux masses (20 min)

Un arbre plein transmet un couple de $500$ N$\cdot$m avec une contrainte de cisaillement admissible de $60$ MPa ; la contrainte vaut $\tau = 16\,T/(\pi\,d^3)$. Résolvez en lettres pour $d$, puis chiffrez le diamètre, avec le couple en N$\cdot$mm. Deux masses $m_1$ et $m_2$ sont reliées entre elles par un ressort $k_2$, et la première au bâti par un ressort $k_1$ : leurs matrices sont $M = \mathrm{diag}(m_1, m_2)$ et $K = \begin{pmatrix} k_1 + k_2 & -k_2 \\ -k_2 & k_2 \end{pmatrix}$. Écrivez l'équation caractéristique $\det(K - \lambda M) = 0$, avec $\lambda = \omega^2$, puis les deux fréquences propres $f = \sqrt{\lambda}/(2\pi)$ pour $m_1 = 5$ kg, $m_2 = 3$ kg, $k_1 = 2$ MN/m et $k_2 = 1$ MN/m.

**Résultat attendu.** Diamètre de 34,88 mm ; fréquences de 66,78 Hz et 138,5 Hz ; équation caractéristique $k_{1} k_{2} + \lambda^{2} m_{1} m_{2} + \lambda \left(- k_{1} m_{2} - k_{2} m_{1} - k_{2} m_{2}\right)= 0$.

**Seconde voie.** La formule fermée $d = \bigl(16\,T/(\pi\,\tau)\bigr)^{1/3}$ redonne 34,8816 mm, et la formule du second degré appliquée aux trois coefficients du polynôme redonne les deux racines en $\lambda$.

**Indication.** Résolvez `Eq(16*T/(pi*d**3), tau)` pour `d` avec des symboles positifs, puis `subs` avec $T = 500\,000$ N$\cdot$mm. Pour les masses, `Matrix` et `diag` donnent $K$ et $M$ ; le déterminant de $K - \lambda M$, développé, est le polynôme demandé. Ses racines en $\lambda$ sont à remplacer par les valeurs numériques, puis à passer à la racine carrée et à diviser par $2\pi$.

In [ ]:
# Votre code :
assert sp.simplify(d_litteral**3 - 16 * T / (sp.pi * tau)) == 0
assert abs(d_arbre_mm - (16 * T_Nmm / (math.pi * tau_adm_MPa))**(1 / 3)) < 1e-12
assert abs(d_arbre_mm - 34.88) < 0.005, "diamètre attendu : 34,88 mm"
# Votre code :
assert sp.simplify(RA_litterale - F * (L - a) / L) == 0
assert sp.simplify(RB_litterale - F * a / L) == 0
assert RA_litterale.subs({F: 2000, L: 1000, a: 500}) == 1000
# Votre code :
assert sp.expand(polynome - polynome_attendu) == 0, "équation à deux degrés de liberté"
# Votre code :
assert len(racines) == 2 and abs(frequences_Hz[0] - 66.78) < 0.005
assert abs(frequences_Hz[1] - 138.5) < 0.05
# Votre code :
assert sp.simplify(decrement_litteral - 2 * sp.pi * alpha / wd) == 0, "2 pi alpha / omega_d"
# Votre code :
assert abs(decrement - 2 * math.pi * zeta / math.sqrt(1 - zeta**2)) < 1e-12
assert abs(decrement - 0.3146) < 5e-5 and abs(f0_Hz - 10.07) < 0.005
print("Exercice 10.2 : contrôles passés")

## Exercice 10.3 : Fréquences propres et treillis (défi, 30 min)

*Première partie.* Reprenez l'équation caractéristique des deux masses de l'exercice 10.2 en gardant $k_2$ comme lettre. Résolvez-la pour $\lambda$, formez les deux fréquences $f = \sqrt{\lambda}/(2\pi)$, changez-les en une fonction avec `lambdify`, et calculez les deux fréquences pour $k_2 = 0{,}5$, $1$ et $2$ MN/m ($m_1 = 5$ kg, $m_2 = 3$ kg, $k_1 = 2$ MN/m).

*Seconde partie.* Un treillis de Warren isostatique compte cinq nœuds, de coordonnées en millimètres $0 : (0\,;\,0)$, $1 : (2\,000\,;\,0)$, $2 : (4\,000\,;\,0)$, $3 : (1\,000\,;\,1\,500)$, $4 : (3\,000\,;\,1\,500)$, et sept barres, dans l'ordre $0$-$1$, $1$-$2$, $3$-$4$, $0$-$3$, $3$-$1$, $1$-$4$, $4$-$2$. Il est articulé au nœud $0$ (deux réactions) et appuyé verticalement au nœud $2$ (une réaction) ; une charge de $20$ kN vers le bas s'applique au nœud $1$. Écrivez l'équilibre des cinq nœuds, soit dix équations, pour les sept efforts normaux (positifs en traction) et les trois réactions, et résolvez le système avec `linsolve`. Quelles barres sont comprimées ?

**Résultat attendu.** Pour $k_2 = 1$ MN/m, 66,78 Hz et 138,5 Hz, les deux fréquences croissant avec $k_2$ (figure du chapitre) ; efforts en kN, dans l'ordre des barres : 6,67, 6,67, $-13{,}33$, $-12{,}02$, 12,02, 12,02 et $-12{,}02$ ; trois barres comprimées.

**Seconde voie.** Par symétrie, chaque appui reprend la moitié de la charge ; la barre $0$-$3$ projette son effort sur la verticale avec le facteur $1\,500/\sqrt{1\,000^2 + 1\,500^2}$ et équilibre cette réaction, d'où un effort de même valeur absolue que celui de la barre $3$-$1$, 12,02 kN.

**Indication.** Première partie : `solve` sur le polynôme en $\lambda$ rend deux racines ; la racine carrée divisée par $2\pi$ donne les fréquences, que `lambdify` change en fonction de $(m_1, m_2, k_1, k_2)$. Seconde partie : pour chaque barre, la composante de l'effort sur chaque axe s'ajoute au nœud d'un bout et se retranche à l'autre ; les réactions et la charge complètent les équations. `linsolve` rend les sept efforts et les trois réactions ; les barres $3$-$4$, $0$-$3$ et $4$-$2$ sont comprimées.

In [ ]:
# Votre code :
assert abs(fleche_np(2000.0, 1000.0, 210000.0, 20.0, 40.0) - fleche_mm) < 1e-12
assert np.allclose(np.round(fleches_mm, 2), [4.41, 1.86, 0.95, 0.55])      # valeurs du chapitre 3
# Votre code :
assert abs(kt_035 - 2.2746) < 5e-5 and abs(kt_035_lineaire - 2.2795) < 5e-5
# Votre code :
assert abs(f1_nombre - frequences_Hz[0]) < 1e-9 and abs(f2_nombre - frequences_Hz[1]) < 1e-9
# Votre code :
assert intensites == [6.67, 6.67, 13.33, 12.02, 12.02, 12.02, 12.02]
assert abs(sum(efforts_kN[:2]) / 2 - 6.6667) < 1e-3 and abs(solution_treillis[7]) < 1e-9
assert abs(fleche_treillis_mm + 1.25) < 0.005, "flèche du treillis attendue : -1,25 mm"
print("Exercice 10.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F10-1 : Équations symboliques (`sympy.solve`)

In [ ]:
import sympy as sp

t, N, Mt, r, se = sp.symbols("t N M_t r sigma_e", positive=True)
sigma = N / (2 * sp.pi * r * t)                 # contrainte normale
tau = Mt / (2 * sp.pi * r**2 * t)               # contrainte de cisaillement
equation = sp.Eq(sigma**2 + 3 * tau**2, se**2)  # critère de von Mises
solution = sp.solve(equation, t)[0]             # isoler l'épaisseur t

cas = {N: 50e3, Mt: 200, r: 0.02, se: 200e6}    # N, N.m, m, Pa
t_num = float(solution.subs(cas)) * 1e3         # conversion en mm
print(f"Épaisseur minimale : {t_num:.2f} mm".replace(".", ","))
controle = sp.simplify(equation.lhs.subs(t, solution) - se**2)
print(f"Équilibre exact : {controle == 0}")

### Fiche F10-2 : Du symbole au tableau (`sympy.lambdify`)

In [ ]:
import numpy as np
import sympy as sp

T, L, G, D, d = sp.symbols("T L G D d", positive=True)
theta = 32 * T * L / (sp.pi * G * (D**4 - d**4))          # angle de torsion
f_theta = sp.lambdify((T, L, G, D, d), theta, "numpy")
d_int = np.array([0.02, 0.03, 0.04])                      # alésages, m
angles = f_theta(500.0, 1.2, 80e9, 0.05, d_int)           # en radians
print(" ".join(f"{x:.3f}" for x in angles * 180 / np.pi).replace(".", ","))
J_np = np.pi * (0.05**4 - d_int**4) / 32
print(np.allclose(angles, 500.0 * 1.2 / (80e9 * J_np)))   # contrôle